# 01. Ingestão de Dados, Qualidade e Camada Silver
**Artefato Analítico-Preditivo para Planejamento de Demanda e Mitigação de Desperdício**  
*Mestrado Profissional em Educação Profissional e Tecnológica (ProfEPT) / Gestão Pública*  
**Pesquisador:** Michel Silva de Souza  

---

## Storytelling: Da Catraca do Refeitório à Tabela Fato Diária
No contexto da escola pública, cada dado reflete uma rotina real: alunos chegando para almoçar, a nutricionista planejando compras quinzenais e preparos diários, e eventos institucionais alterando o fluxo de estudantes.

Neste primeiro notebook do artefato, nós executamos:
1. **Ingestão Auditada da Camada Bronze:** Calendário, Consumo, Reservas/Planejamento, Cardápio e Clima.
2. **Higienização e Padronização:** Anonimização de siglas sensíveis, normalização textual de cardápio e separação de regimes (2023-2025: Reservas Discentes; 2026: Planejamento Nutricionista).
3. **Detecção Determinística de Dias Atípicos:** Separação causal de outliers, greves e efeitos de fim de semestre, sem vazamento temporal.
4. **Consolidação da Camada Silver:** Criação da `silver_fato_diaria.parquet` com rastreabilidade total.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from src.core.config import BRONZE, SILVER
from src.core.bronze_silver import executar_pipeline_silver
from src.core.terminology import NOMES_TRATAMENTOS

# Estilo visual para gráficos da dissertação
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 11
plt.rcParams['figure.dpi'] = 120


### 1. Execução do Pipeline Bronze -> Silver
Executamos as transformações automatizadas encapsuladas no módulo `src.core.bronze_silver`.

In [ ]:
relatorio = executar_pipeline_silver()
pd.DataFrame([relatorio]).T.rename(columns={0: "Valor Operacional"})


### 2. Análise da Tabela Fato Diária Integrada
A tabela `silver_fato_diaria` reúne em uma única linha por data todas as dimensões do restaurante institucional.

In [ ]:
fato = pd.read_parquet(SILVER / "silver_fato_diaria.parquet")
print(f"Dimensões da Fato Diária: {fato.shape[0]} dias x {fato.shape[1]} colunas")
fato.head(5)


### 3. Storytelling dos Regimes de Planejamento (2023 a 2026)
Um marco essencial da pesquisa é a distinção de regimes:
* **2023 a 2025:** Existia um sistema de reservas individuais feitas pelos próprios alunos (`reserva_discente`).
* **2026 em diante:** A nutricionista assumiu a estimativa centralizada de quantitativo (`plano_nutricionista`).


In [ ]:
regimes = fato[fato["tem_consumo"]].groupby("ano")[["reserva_discente", "plano_nutricionista", "consumo"]].count()
regimes.rename(columns={
    "reserva_discente": "Dias com Reserva Discente",
    "plano_nutricionista": "Dias com Plano Nutricionista",
    "consumo": "Dias com Refeições Servidas"
})


### 4. Diagnóstico e Tratamento Determinístico de Dias Atípicos
Para que o aprendizado de máquina aprenda a dinâmica real de consumo sem sofrer distorção por eventos pontuais (enchentes, greves, quedas severas com < 30 refeições), os dias são classificados com regras auditáveis baseadas na distância para o nível basal recente.

In [ ]:
dias = pd.read_parquet(SILVER / "silver_dias_classificados.parquet")
dist_tratamento = dias["tratamento"].value_counts().reset_index()
dist_tratamento["nome_operacional"] = dist_tratamento["tratamento"].map(NOMES_TRATAMENTOS)
dist_tratamento["elegivel_treino"] = dist_tratamento["tratamento"].isin(["normal", "alto_mantido", "previsivel_calendario"])
dist_tratamento.columns = ["Tratamento Técnico", "Dias", "Nome Operacional", "Elegível Treino"]
dist_tratamento


### 5. Visualização Gráfica: Série Histórica com Identificação de Anomalias
Este gráfico ilustra a série temporal de consumo escolar, permitindo identificar visualmente os períodos de greve e outliers isolados de treinamento.

In [ ]:
servidos = fato[fato["tem_consumo"]].copy().sort_values("data")
servidos = servidos.merge(dias[["data", "tratamento", "referencia"]], on="data", how="left")

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(servidos["data"], servidos["consumo"], color="#2b5c8f", alpha=0.6, label="Consumo Real (Refeições)")
ax.plot(servidos["data"], servidos["referencia"], color="#e67e22", linestyle="--", linewidth=1.5, label="Nível Basal Causal (Referência)")

# Destacar outliers isolados do treino
outliers = servidos[~servidos["tratamento"].isin(["normal", "alto_mantido", "previsivel_calendario"])]
ax.scatter(outliers["data"], outliers["consumo"], color="#e74c3c", s=40, zorder=5, label="Dias Atípicos Isolados do Treino")

ax.set_title("Evolução Temporal do Consumo de Refeições e Nível Basal (2023 - 2026)", fontsize=13, fontweight="bold", pad=12)
ax.set_xlabel("Data")
ax.set_ylabel("Quantidade de Refeições")
ax.legend(loc="upper right", frameon=True)
plt.tight_layout()
plt.show()
